# Part 6 — Sanity checks

A consensus hub can be real, or an artefact of the prior giving high-degree nodes
more chances to appear. Three controls test the second possibility: a Spearman
check of consensus frequency against prior degree (primary and pooled cohorts), a
random patient-set null, and an optional annotation-based pathway screen. At UC1
cohort sizes the per-cohort test is power-limited — the pooled and null rows carry
more weight. None of this claims biological discovery.

In [ ]:
import pickle

import numpy as np
import pandas as pd
from scipy.stats import fisher_exact, spearmanr
from IPython.display import display

from uc1_common import *  # noqa: F403

primary_patient_networks = load_carnival()[PRIMARY_LAMBDA]
cohort_definitions = load_cohorts()
consensus_outputs = pickle.loads(CONSENSUS_PKL.read_bytes())
G = load()

pkn_deg = pkn_degree(pkn_edges(G))
primary_node_df = consensus_outputs[PRIMARY_COHORT_LABEL]['node_df']
primary_consensus_selected = primary_node_df[primary_node_df['selected_for_consensus']].copy()

## Degree-bias and random patient-set null

In [ ]:
def with_degree(df):
    return df.merge(pkn_deg.rename_axis('vertex_id').reset_index(), on='vertex_id', how='left').fillna({'pkn_degree': 0})


def selected(cohort_label):
    df = consensus_outputs[cohort_label]['node_df']
    return df[df['selected_for_consensus']]


consensus_with_degree = with_degree(primary_consensus_selected)
degree_rho, degree_p = (spearmanr(consensus_with_degree['patient_frequency'], consensus_with_degree['pkn_degree'])
                        if len(consensus_with_degree) >= 2 else (np.nan, np.nan))

# Pooled across sensitivity cohorts: dedup consensus nodes, take max frequency.
pooled_rows = [selected(c)[['vertex_id', 'patient_frequency']] for c in SENSITIVITY_COHORT_LABELS if not selected(c).empty]
if pooled_rows:
    pooled = with_degree(pd.concat(pooled_rows, ignore_index=True)
                         .groupby('vertex_id', as_index=False)['patient_frequency'].max())
    pooled_rho, pooled_p = (spearmanr(pooled['patient_frequency'], pooled['pkn_degree']) if len(pooled) >= 2 else (np.nan, np.nan))
    pooled_n = len(pooled)
else:
    pooled_rho, pooled_p, pooled_n = np.nan, np.nan, 0

bias_summary = pd.DataFrame({
    'metric': ['primary_n', 'primary_spearman_rho', 'primary_spearman_pvalue',
               'pooled_n', 'pooled_spearman_rho', 'pooled_spearman_pvalue'],
    'value': [len(consensus_with_degree), degree_rho, degree_p, pooled_n, pooled_rho, pooled_p],
})
bias_summary.to_csv(TABLES / 'degree_bias_summary.csv', index=False)
consensus_with_degree.to_csv(TABLES / 'consensus_nodes_with_degree.csv', index=False)

solved_primary = [p for p in cohort_definitions[PRIMARY_COHORT_LABEL]
                  if p in primary_patient_networks and primary_patient_networks[p]['solver_ok']]
solved_pool = [p for p, rec in primary_patient_networks.items() if rec['solver_ok']]
rng = np.random.default_rng(SEED)
if solved_primary and len(solved_pool) >= len(solved_primary):
    null_rows = []
    for repeat in range(NULL_REPEATS):
        sampled = rng.choice(solved_pool, size=len(solved_primary), replace=False)
        sample_nodes, _, _ = build_consensus_layer({p: primary_patient_networks[p] for p in sampled}, CONSENSUS_MIN_FREQ)
        sel_deg = with_degree(sample_nodes[sample_nodes['selected_for_consensus']])
        null_rows.append({'repeat': repeat, 'consensus_size': int(len(sel_deg)),
                          'mean_pkn_degree': float(sel_deg['pkn_degree'].mean()) if not sel_deg.empty else np.nan})
    null_df = pd.DataFrame(null_rows)
    null_df.to_csv(TABLES / 'null_patient_set_summary.csv', index=False)
    null_summary = pd.DataFrame({
        'metric': ['observed_consensus_size', 'null_mean_consensus_size', 'observed_mean_degree', 'null_mean_degree'],
        'value': [int(len(primary_consensus_selected)), float(null_df['consensus_size'].mean()),
                  float(consensus_with_degree['pkn_degree'].mean()) if not consensus_with_degree.empty else np.nan,
                  float(null_df['mean_pkn_degree'].mean())]})
else:
    null_summary = pd.DataFrame({'metric': ['null_model_note'],
                                 'value': ['insufficient solved patients for random patient-set null model']})
null_summary.to_csv(TABLES / 'null_summary.csv', index=False)

print('Degree-bias check (primary vs pooled cohorts):')
display(bias_summary)
print('\nRandom patient-set null model:')
null_summary

### Optional annotation-based pathway screen

When the imported vertex annotations carry a pathway-like column, run a Fisher's
exact over-representation screen on the consensus set — a lightweight sanity
check, not a pathway-discovery pipeline.

In [ ]:
vertex_attr_df = G.vertex_attributes.to_pandas()
pathway_cols = [c for c in vertex_attr_df.columns if 'pathway' in c.lower()]


def split_terms(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    text = str(value)
    for sep in ['|', ';', ',']:
        text = text.replace(sep, ';')
    return [t.strip() for t in text.split(';') if t.strip() and t.strip().lower() != 'nan']


if pathway_cols and not primary_consensus_selected.empty:
    col = pathway_cols[0]
    background = vertex_attr_df[['vertex_id', col]].dropna()
    selected_ids = set(primary_consensus_selected['vertex_id'])
    universe = set(background['vertex_id'].astype(str))
    selected_universe = universe & selected_ids

    bg_ids, sel_ids = {}, {}
    # itertuples(name=None) -> plain tuples, so ':' in column names is safe.
    for vid, raw in background.itertuples(index=False, name=None):
        vid = str(vid)
        for term in set(split_terms(raw)):
            bg_ids.setdefault(term, set()).add(vid)
            if vid in selected_ids:
                sel_ids.setdefault(term, set()).add(vid)

    rows = []
    for term, bg in bg_ids.items():
        a = len(sel_ids.get(term, set()))
        if a == 0:
            continue
        b = len(selected_universe) - a
        c = len(bg - selected_universe)
        d = len(universe - selected_universe) - c
        _, pvalue = fisher_exact([[a, b], [c, d]], alternative='greater')
        rows.append({'annotation_column': col, 'term': term, 'selected_count': a,
                     'background_count': len(bg), 'pvalue': pvalue})
    annotation_enrichment = pd.DataFrame(rows).sort_values(['pvalue', 'selected_count']).head(20)
    annotation_enrichment.to_csv(TABLES / 'annotation_pathway_screen.csv', index=False)
else:
    annotation_enrichment = pd.DataFrame({'note': ['No pathway-like vertex annotation column was available.']})

annotation_enrichment.head(20)

## Cohort sensitivity — top-10% vs top-20%

In [ ]:
cohort_sensitivity = pd.DataFrame([
    {'cohort_label': label, 'n_patients': payload['summary']['n_patients'],
     'n_consensus_nodes': payload['summary']['n_consensus_nodes'],
     'median_node_frequency': float(sel['patient_frequency'].median()) if not sel.empty else np.nan,
     'mean_abs_signal': float(sel['mean_signal'].abs().mean()) if not sel.empty else np.nan}
    for label, payload in consensus_outputs.items()
    for sel in [payload['node_df'][payload['node_df']['selected_for_consensus']]]
])
cohort_sensitivity.to_csv(TABLES / 'cohort_sensitivity.csv', index=False)
cohort_sensitivity